# TorchInspector - Colab Demo

Real-time training monitoring: loss curves, activation stats, feature maps, Grad-CAM heatmaps, weight heatmaps, health reports.

In [ ]:
# 1. Clone and install
!git clone https://github.com/blackcat312340/torchinspector.git
%cd torchinspector
!pip install -e . -q

In [ ]:
# 2. Verify import
from torchinspector import Inspector
import torch
print(f"PyTorch {torch.__version__}")
print("TorchInspector OK")

In [ ]:
# 3. Load TensorBoard + clean old logs
%load_ext tensorboard
!rm -rf runs/mnist_demo

In [ ]:
# 4. Train CNN on MNIST with full monitoring
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

model = nn.Sequential(
    nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Flatten(),
    nn.Linear(32 * 7 * 7, 10),
)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

transform = transforms.Compose([transforms.ToTensor()])
train_ds = datasets.MNIST('data', train=True, download=True, transform=transform)
loader = DataLoader(train_ds, batch_size=64, shuffle=True)

ins = Inspector(model, opt, 'runs/mnist_demo',
                log_interval=50,
                feature_map_interval=200,
                feature_map_channels=8,
                weight_heatmap_interval=200,
                health_report_interval=200)

watched = ins.watch_auto(max_layers=6)
print(f"Auto-watching: {watched}")

step = 0
for epoch in range(2):
    for x, y in loader:
        opt.zero_grad()
        loss = nn.functional.cross_entropy(model(x), y)
        loss.backward()
        opt.step()
        step += 1
        ins.step(loss=loss.item())

        if step % 500 == 0 and step > 0:
            ins.explain(x[:1], method='gradcam', target=y[0].item())
            print(f'Step {step}: Grad-CAM on digit {y[0].item()}')

ins.close()
print(f'Done! {step} steps')

In [ ]:
# 5. Launch TensorBoard
%tensorboard --logdir=runs/mnist_demo

## What you see

**Scalars**
- `train/loss` - loss curve
- `activations/*/dead_neuron_ratio` - dead neuron ratio per layer
- `features/*/dead_filter_count` - dead conv filter count
- `bn/*` - BatchNorm running stats
- `pool/*` - Pooling stats

**Images**
- `features/0/channels` - Conv layer 1 feature maps (8 channels)
- `features/3/channels` - Conv layer 2 feature maps
- `weights/*/matrix` - Weight matrix heatmaps
- `explain/*/gradcam` - Grad-CAM heatmaps

**Health Reports** (in cell output)
- Printed every 200 steps to stderr
- Shows loss trend, active alerts, one-line summary